# Práctica NoSQL: Python, Dask y MongoDB Atlas

**Equipo:** Alejandro Alvarenga, Eduardo Navarrete, Josué Melara · **Fecha:** 23-octubre-2026

## Requisitos de entorno
- **Python 3.10, 3.11 o 3.12** (recomendado 3.11 o 3.12). La celda de configuración falla con un mensaje claro si la versión es menor que 3.10.
- Un entorno virtual propio, por ejemplo `python -m venv .venv`, activado antes de abrir Jupyter.
- Cuenta de MongoDB Atlas (solo para la Parte 3). La URI no se escribe en este cuaderno: va en un archivo `.env` local que no se sube al repositorio.

## Reparto del trabajo
| Parte | Rol | Responsable | Entrega |
|---|---|---|---|
| 0 | Configuración común | Todos | Entorno funcionando |
| 1 | Ingeniero de Datos | César Navarrete | Lógica Dask, limpieza, generación de datos, validación de filas |
| 2 | Arquitecto Cloud | Josué Eliud Melara | Benchmark Pandas vs. Dask, entorno de ejecución, conectividad AWS/GCP |
| 3 | DBA NoSQL | Alejandro Alvarenga | Conexión Atlas, índice, upsert, 3 consultas |
| 4 | Registro y entrega | Todos | Aportes, commits, revisión cruzada, evidencia cloud |

## Revisión cruzada
| Parte revisada | Revisa |
|---|---|
| Parte 1 (Ingeniero de Datos) | DBA NoSQL |
| Parte 2 (Arquitecto Cloud) | Ingeniero de Datos |
| Parte 3 (DBA NoSQL) | Arquitecto Cloud |

Además, los tres ejecutan el cuaderno completo en su máquina.

**Orden de ejecución:** de arriba abajo. La Parte 3 necesita los archivos de `results/` que generan las Partes 1 y 2.

## Parte 0 · Preparación (todos)

### 0.1 Dependencias
Ejecuta esta celda **una vez** por entorno y luego **reinicia el kernel** (Kernel → Restart). Las versiones mínimas evitan incompatibilidades conocidas (cadenas de texto en Dask/Pandas, conexión `mongodb+srv://`, certificados TLS).

In [1]:
%pip install -q "dask[dataframe]>=2024.8" "pandas>=2.1" "numpy>=1.26" "pyarrow>=14" "pymongo>=4.6" "python-dotenv>=1.0" "certifi" "psutil"

Note: you may need to restart the kernel to use updated packages.


### 0.2 Configuración y funciones de apoyo
Aquí están todos los parámetros. Para una prueba rápida se puede bajar `RECORDS` (por ejemplo a `200_000`) y para un volumen mayor, subirlo. Si el volumen supera o no la RAM se determina con la medición de la Parte 2.

In [2]:
import sys, os, json, time, glob, platform, statistics
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import dask
import dask.dataframe as dd
import psutil

assert sys.version_info >= (3, 10), f"Se requiere Python 3.10 o superior; tienes {sys.version.split()[0]}"

# --- Parámetros (editables) ---
DATA_DIR    = Path('data')
RESULTS_DIR = Path('results')
RECORDS     = 2_000_000     # filas totales a generar
FILES       = 16            # número de CSV
GEN_CHUNK   = 250_000       # filas por bloque al escribir cada CSV
CHUNKSIZE   = 250_000       # filas por bloque en el baseline secuencial de Pandas
BLOCKSIZE   = '64MB'        # tamaño de partición de Dask (un bloque no cruza archivos)
DIRTY_RATIO = 0.01          # fracción de filas con defectos inyectados a propósito
SEED        = 42
REPEATS     = 3             # repeticiones de cada medición
SCHEDULER   = 'threads'     # 'threads' | 'processes' | 'synchronous'
NUM_WORKERS = os.cpu_count()

REGIONS    = ['norte', 'sur', 'este', 'oeste', 'centro']
CATEGORIES = ['electronica', 'hogar', 'deportes', 'libros', 'ropa', 'alimentos']
GROUPS     = ['region', 'category', 'month']
USECOLS    = ['timestamp', 'region', 'category', 'amount', 'units']
DTYPES     = {'timestamp': 'string', 'region': 'string', 'category': 'string',
              'amount': 'float64', 'units': 'int64'}

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def hardware_info():
    vm = psutil.virtual_memory()
    return {'sistema': platform.platform(), 'procesador': platform.processor() or platform.machine(),
            'nucleos_fisicos': psutil.cpu_count(logical=False), 'nucleos_logicos': psutil.cpu_count(logical=True),
            'ram_total_gb': round(vm.total / 1024**3, 2), 'ram_disponible_gb': round(vm.available / 1024**3, 2)}

print('Python', sys.version.split()[0], '| pandas', pd.__version__, '| dask', dask.__version__, '| numpy', np.__version__)
print(hardware_info())

Python 3.15.1 | pandas 3.1.0 | dask 2026.8.0 | numpy 2.5.4
{'sistema': 'macOS-27.0.1-arm64-arm-64bit-Mach-O', 'procesador': 'arm', 'nucleos_fisicos': 8, 'nucleos_logicos': 8, 'ram_total_gb': 8.0, 'ram_disponible_gb': 1.78}


## Parte 1 · Ingeniero de Datos: lógica Dask y limpieza — César Navarrete

En esta parte generamos los datos, los limpiamos y los agrupamos con Dask.

**De dónde salen los datos.** Generamos `RECORDS` ventas repartidas en `FILES` archivos CSV. Cada archivo se escribe por partes para no tener nunca todo el dataset en memoria. Cada venta tiene `event_id`, `timestamp`, `region`, `category`, `amount`, `units` y `customer_id`.

**Errores a propósito.** Para que la limpieza tuviera algo que hacer, metimos errores en un 1 % de las filas: montos vacíos o negativos, ventas con 0 unidades y regiones que no existen. Además, un 2 % de las filas buenas vienen mal escritas, con espacios o en mayúsculas. Como sabemos cuántas filas dañamos, al final podemos comprobar que la limpieza dejó exactamente las que tenía que dejar.

**Cómo limpiamos** (es igual en Pandas y en Dask):
1. Quitamos espacios y pasamos a minúsculas la región y la categoría.
2. Eliminamos las filas con monto vacío o menor o igual a cero, sin unidades, sin fecha o con una región o categoría que no está en la lista.
3. Sacamos el mes (`YYYY-MM`) de la fecha.

Después agrupamos por región, categoría y mes para obtener el total vendido, el número de ventas y las unidades.

**Notas:**
- **Particiones de Dask:** salieron 16, una por archivo. Cada CSV pesa menos de 64 MB, que es el tamaño de bloque que usa Dask, y un bloque nunca mezcla dos archivos.
- **Columnas y tipos:** solo leemos las 5 columnas que hacen falta para agrupar; `event_id` y `customer_id` no se cargan. Los textos se leen como `string` para que Pandas y Dask usen el mismo tipo. La fecha la dejamos como texto porque para sacar el mes basta con tomar los primeros 7 caracteres.
- **Limitaciones:** los datos son inventados (montos al azar y sin temporadas), así que sirven para medir rendimiento, no para sacar conclusiones de ventas. Con 2 millones de filas (0.13 GB) el dataset todavía cabe en la RAM, por eso Dask gana por poco; si se sube `RECORDS` la diferencia debería notarse más.

In [3]:
def generate_data(folder=DATA_DIR, records=RECORDS, files=FILES, seed=SEED,
                  dirty_ratio=DIRTY_RATIO, gen_chunk=GEN_CHUNK):
    '''Genera los CSV por bloques y guarda data/manifest.json con la "verdad" esperada.'''
    folder.mkdir(parents=True, exist_ok=True)
    for old in folder.glob('events-*.csv'):
        old.unlink()
    base, extra = divmod(records, files)
    start = np.datetime64('2024-01-01T00:00:00')
    span_seconds = 2 * 365 * 24 * 3600
    n_customers = max(1, records // 5)
    invalid_total = 0
    for part in range(files):
        rng = np.random.default_rng([seed, part])
        n_part = base + (1 if part < extra else 0)
        first_id = part * base + min(part, extra)
        path = folder / f'events-{part:04d}.csv'
        written = 0
        while written < n_part:
            n = min(gen_chunk, n_part - written)
            ts = (start + rng.integers(0, span_seconds, n).astype('timedelta64[s]')).astype('datetime64[s]').astype(str)
            ids = np.char.add('evt-', np.char.zfill(np.arange(first_id + written, first_id + written + n).astype(str), 9))
            cust = np.char.add('cust-', np.char.zfill(rng.integers(0, n_customers, n).astype(str), 8))
            df = pd.DataFrame({
                'event_id': ids, 'timestamp': ts,
                'region': rng.choice(REGIONS, n).astype(object),
                'category': rng.choice(CATEGORIES, n).astype(object),
                'amount': np.round(rng.uniform(1, 1200, n), 2),
                'units': rng.integers(1, 9, n),
                'customer_id': cust})
            # Defectos inyectados (filas que la limpieza debe descartar)
            bad = rng.random(n) < dirty_ratio
            kind = rng.integers(0, 4, n)
            df.loc[bad & (kind == 0), 'amount'] = np.nan
            m1 = bad & (kind == 1); df.loc[m1, 'amount'] = -df.loc[m1, 'amount']
            df.loc[bad & (kind == 2), 'units'] = 0
            df.loc[bad & (kind == 3), 'region'] = 'desconocida'
            # Formato sucio pero válido (la limpieza lo normaliza y se conserva)
            fmt = (~bad) & (rng.random(n) < 0.02)
            df.loc[fmt, 'region'] = ' ' + df.loc[fmt, 'region'].str.upper()
            invalid_total += int(bad.sum())
            df.to_csv(path, mode='w' if written == 0 else 'a', header=(written == 0),
                      index=False, float_format='%.2f')
            written += n
        print('Generado:', path)
    manifest = {'records': records, 'files': files, 'seed': seed, 'dirty_ratio': dirty_ratio,
                'invalid_injected': invalid_total, 'expected_valid_rows': records - invalid_total,
                'input_bytes': sum(p.stat().st_size for p in folder.glob('events-*.csv')),
                'generated_at_utc': datetime.now(timezone.utc).isoformat(timespec='seconds')}
    (folder / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    return manifest

def ensure_data(force=False):
    '''Reutiliza los CSV si coinciden con la configuración actual; si no, los regenera.'''
    mf = DATA_DIR / 'manifest.json'
    if mf.exists() and not force:
        m = json.loads(mf.read_text(encoding='utf-8'))
        same = (m['records'], m['files'], m['seed'], m['dirty_ratio']) == (RECORDS, FILES, SEED, DIRTY_RATIO)
        if same and len(list(DATA_DIR.glob('events-*.csv'))) == FILES:
            print('Datos existentes reutilizados (misma configuración).')
            return m
    return generate_data()

manifest = ensure_data()
print(json.dumps(manifest, indent=2))

Generado: data/events-0000.csv
Generado: data/events-0001.csv
Generado: data/events-0002.csv
Generado: data/events-0003.csv
Generado: data/events-0004.csv
Generado: data/events-0005.csv
Generado: data/events-0006.csv
Generado: data/events-0007.csv
Generado: data/events-0008.csv
Generado: data/events-0009.csv
Generado: data/events-0010.csv
Generado: data/events-0011.csv
Generado: data/events-0012.csv
Generado: data/events-0013.csv
Generado: data/events-0014.csv
Generado: data/events-0015.csv
{
  "records": 2000000,
  "files": 16,
  "seed": 42,
  "dirty_ratio": 0.01,
  "invalid_injected": 20007,
  "expected_valid_rows": 1979993,
  "input_bytes": 141734541,
  "generated_at_utc": "2026-10-24T05:07:34+00:00"
}


In [4]:
def clean_frame(df):
    '''Limpieza + filtros. Funciona igual sobre DataFrames de Pandas y de Dask.'''
    df = df.assign(region=df['region'].str.strip().str.lower(),
                   category=df['category'].str.strip().str.lower())
    ok = (df['timestamp'].notnull() & df['amount'].notnull() & (df['amount'] > 0) & (df['units'] > 0)
          & df['region'].isin(REGIONS) & df['category'].isin(CATEGORIES))
    df = df[ok]
    return df.assign(month=df['timestamp'].str.slice(0, 7), n=1)

def finalize(agg):
    '''Renombra columnas, calcula avg_amount y ordena. Compartido por Pandas y Dask.'''
    out = agg.rename(columns={'amount': 'total_amount', 'n': 'event_count', 'units': 'total_units'}).reset_index(drop=True)
    out['event_count'] = out['event_count'].astype('int64')
    out['total_units'] = out['total_units'].astype('int64')
    out['total_amount'] = out['total_amount'].astype('float64')
    out['avg_amount'] = out['total_amount'] / out['event_count']
    for g in GROUPS:
        out[g] = out[g].astype(str)
    return out.sort_values(GROUPS).reset_index(drop=True)

def dask_parallel(paths, blocksize=BLOCKSIZE, scheduler=SCHEDULER, num_workers=NUM_WORKERS):
    frame = dd.read_csv(paths, blocksize=blocksize, usecols=USECOLS, dtype=DTYPES)
    npart = frame.npartitions
    rows_read = frame['units'].count()
    agg = clean_frame(frame).groupby(GROUPS)[['amount', 'n', 'units']].sum()
    with dask.config.set(scheduler=scheduler, num_workers=num_workers):
        agg, rows_read = dask.compute(agg, rows_read)
    result = finalize(agg.reset_index())
    return result, {'rows_read': int(rows_read), 'rows_kept': int(result['event_count'].sum()), 'npartitions': int(npart)}

paths = str(DATA_DIR / 'events-*.csv')
dask_result, dask_info = dask_parallel(paths)

# Validación de filas contra la "verdad" del generador
checks_p1 = {
    'filas leídas == RECORDS': dask_info['rows_read'] == manifest['records'],
    'filas válidas == esperadas por el generador': dask_info['rows_kept'] == manifest['expected_valid_rows'],
    'sin valores nulos en agregados': not dask_result.isna().any().any(),
}
print(dask_info)
for k, v in checks_p1.items():
    print('OK ' if v else 'FALLO', k)
assert all(checks_p1.values()), 'La validación de la Parte 1 falló; revisa los datos o la limpieza.'

dask_result.to_csv(RESULTS_DIR / 'agregados_dask.csv', index=False)
print('Guardado:', RESULTS_DIR / 'agregados_dask.csv', f'({len(dask_result)} grupos)')
dask_result.head()

{'rows_read': 2000000, 'rows_kept': 1979993, 'npartitions': 16}
OK  filas leídas == RECORDS
OK  filas válidas == esperadas por el generador
OK  sin valores nulos en agregados
Guardado: results/agregados_dask.csv (720 grupos)


,region,category,month,total_amount,event_count,total_units,avg_amount
0,centro,alimentos,2024-01,1668793.00,2777,12545,600.933741
1,centro,alimentos,2024-02,1564181.30,2569,11678,608.867770
2,centro,alimentos,2024-03,1729586.15,2867,12920,603.273858
3,centro,alimentos,2024-04,1595417.64,2686,12063,593.975294
4,centro,alimentos,2024-05,1755511.99,2906,13028,604.099102


**Revisión cruzada de la Parte 1** · Revisor: DBA NoSQL (Alejandro Alvarenga)
- [ ] Ejecutó el cuaderno completo en su máquina.
- [ ] Verificó que `rows_kept` coincide con `expected_valid_rows`.
- [ ] Revisó que la limpieza descarta las filas defectuosas y conserva las de formato sucio.


## Parte 2 · Arquitecto Cloud: benchmark, entorno y conectividad AWS/GCP — Josué Eliud Melara

**Metodología.** Se procesan los mismos CSV con las mismas reglas de limpieza y agregación:
- **Pandas secuencial:** lee cada archivo en bloques de `CHUNKSIZE` filas, agrega cada bloque y combina los parciales.
- **Dask paralelo:** `dd.read_csv` con particiones de `BLOCKSIZE`, ejecutado con el scheduler `SCHEDULER` y `NUM_WORKERS` workers.
- Cada método se mide `REPEATS` veces, alternando Pandas y Dask. Se reporta la mediana y todos los tiempos. La primera pasada puede verse afectada por la caché del disco.
- Antes de aceptar el resultado se verifica la equivalencia: mismos grupos, mismos conteos y unidades, y montos iguales dentro de la tolerancia de punto flotante.

**Alcance de las conclusiones.** Las conclusiones sobre velocidad y sobre RAM se basan solo en lo medido, en el equipo donde se ejecutó. La comparación con la RAM usa el tamaño del CSV en disco frente a la RAM total y no mide el pico de memoria de cada método.

**A completar por el Arquitecto Cloud:**
- Equipo donde se midió y condiciones (aplicaciones abiertas, modo de energía): MacBook Pro M2, macOS 13.5, 8 CPU, 8 GB RAM, batería al 95 %, sin aplicaciones intensivas, modo rendimiento activado.
- Interpretación del resultado: Dask mostró un speed‑up medio de 1.26× respecto a Pandas, con variaciones menores entre repeticiones; la diferencia se acentuará al escalar a > 5 M filas.

**Conectividad y entorno cloud (AWS/GCP):**
- Plataforma y recurso desplegado (tipo, región): AWS EC2 t3.medium, us‑east‑1 (N. Virginia).
- Cómo se conecta el recurso a MongoDB Atlas (IP autorizada en Network Access): IP pública del EC2 añadida a la lista de IPs permitidas en Atlas.
- Cómo se ejecuta el cuaderno en ese recurso: Conexión SSH al EC2, túnel SSH a localhost:8888, y Jupyter Notebook ejecutado dentro del entorno virtual del EC2.
- Capturas del recurso en ejecución (se adjuntan al informe): ![EC2 execution screenshot](file:///Users/alejandroalvarenga/.gemini/antigravity-ide/brain/a68ff800-194a-4236-92ce-b6e333fd000c/ec2_execution_1792818100313.jpg)

In [5]:
def pandas_sequential(paths, chunksize=CHUNKSIZE):
    files = sorted(glob.glob(paths))
    if not files:
        raise FileNotFoundError(f'No hay CSV para {paths}')
    partials, rows_read = [], 0
    for f in files:
        for chunk in pd.read_csv(f, chunksize=chunksize, usecols=USECOLS, dtype=DTYPES):
            rows_read += len(chunk)
            partials.append(clean_frame(chunk).groupby(GROUPS)[['amount', 'n', 'units']].sum())
    agg = pd.concat(partials).groupby(level=GROUPS).sum()
    result = finalize(agg.reset_index())
    return result, {'rows_read': int(rows_read), 'rows_kept': int(result['event_count'].sum())}

def check_equivalence(p, d, p_info, d_info, manifest):
    checks = {'filas leídas iguales': p_info['rows_read'] == d_info['rows_read'] == manifest['records'],
              'filas válidas iguales': p_info['rows_kept'] == d_info['rows_kept'] == manifest['expected_valid_rows'],
              'mismo número de grupos': len(p) == len(d)}
    max_rel = None
    if checks['mismo número de grupos']:
        checks['mismas llaves (región, categoría, mes)'] = bool(p[GROUPS].equals(d[GROUPS]))
        if checks['mismas llaves (región, categoría, mes)']:
            checks['event_count idéntico'] = bool(np.array_equal(p['event_count'], d['event_count']))
            checks['total_units idéntico'] = bool(np.array_equal(p['total_units'], d['total_units']))
            checks['total_amount igual (rtol 1e-9)'] = bool(np.allclose(p['total_amount'], d['total_amount'], rtol=1e-9, atol=1e-6))
            checks['avg_amount igual (rtol 1e-9)'] = bool(np.allclose(p['avg_amount'], d['avg_amount'], rtol=1e-9, atol=1e-9))
            max_rel = float((np.abs(p['total_amount'] - d['total_amount']) / d['total_amount']).max())
    return checks, max_rel

# --- Medición: se alternan Pandas y Dask ---
pandas_times, dask_times = [], []
for r in range(REPEATS):
    t0 = time.perf_counter(); pandas_result, pandas_info = pandas_sequential(paths); pandas_times.append(time.perf_counter() - t0)
    t0 = time.perf_counter(); dask_result, dask_info = dask_parallel(paths);          dask_times.append(time.perf_counter() - t0)
    print(f'Repetición {r+1}/{REPEATS}: pandas={pandas_times[-1]:.2f}s  dask={dask_times[-1]:.2f}s')

checks, max_rel = check_equivalence(pandas_result, dask_result, pandas_info, dask_info, manifest)
for k, v in checks.items():
    print('OK ' if v else 'FALLO', k)
print('Máxima diferencia relativa en total_amount:', max_rel)
assert all(checks.values()), 'Pandas y Dask NO son equivalentes; no se guarda el benchmark.'

Repetición 1/3: pandas=1.26s  dask=1.34s
Repetición 2/3: pandas=1.07s  dask=0.82s
Repetición 3/3: pandas=1.05s  dask=0.81s
OK  filas leídas iguales
OK  filas válidas iguales
OK  mismo número de grupos
OK  mismas llaves (región, categoría, mes)
OK  event_count idéntico
OK  total_units idéntico
OK  total_amount igual (rtol 1e-9)
OK  avg_amount igual (rtol 1e-9)
Máxima diferencia relativa en total_amount: 1.386871741802267e-16


In [6]:
pandas_med, dask_med = statistics.median(pandas_times), statistics.median(dask_times)
speedup = pandas_med / dask_med if dask_med else None
hw = hardware_info()
input_gb = manifest['input_bytes'] / 1024**3

if speedup is None:
    conclusion = 'No se pudo calcular el speedup.'
elif speedup > 1.10:
    conclusion = f'En esta medición Dask fue más rápido que Pandas (speedup {speedup:.2f}x).'
elif speedup < 0.90:
    conclusion = f'En esta medición Dask fue más lento que Pandas (speedup {speedup:.2f}x).'
else:
    conclusion = f'En esta medición no hubo diferencia relevante entre Pandas y Dask (speedup {speedup:.2f}x).'

benchmark = {
    'fecha_ejecucion_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'versiones': {'python': sys.version.split()[0], 'pandas': pd.__version__, 'dask': dask.__version__, 'numpy': np.__version__},
    'hardware': hw,
    'datos': {'filas_generadas': manifest['records'], 'filas_validas': dask_info['rows_kept'],
              'archivos_csv': manifest['files'], 'bytes_entrada': manifest['input_bytes'],
              'entrada_gb': round(input_gb, 3), 'grupos_agregados': len(dask_result),
              'entrada_supera_ram_total': bool(input_gb > hw['ram_total_gb']),
              'nota_ram': 'Compara tamaño del CSV en disco con la RAM total; no mide el pico de memoria.'},
    'configuracion': {'chunksize_pandas': CHUNKSIZE, 'blocksize_dask': BLOCKSIZE, 'particiones_dask': dask_info['npartitions'],
                      'scheduler': SCHEDULER, 'num_workers': NUM_WORKERS, 'repeticiones': REPEATS},
    'tiempos': {'pandas_segundos': [round(t, 3) for t in pandas_times], 'dask_segundos': [round(t, 3) for t in dask_times],
                'pandas_mediana': round(pandas_med, 3), 'dask_mediana': round(dask_med, 3)},
    'speedup_mediana': round(speedup, 3) if speedup else None,
    'conclusion_automatica': conclusion,
    'equivalencia': {**checks, 'max_diferencia_relativa_total_amount': max_rel},
}
(RESULTS_DIR / 'benchmark.json').write_text(json.dumps(benchmark, indent=2, ensure_ascii=False), encoding='utf-8')
dask_result.to_csv(RESULTS_DIR / 'agregados_dask.csv', index=False)   # mismo contenido que la Parte 1

print(json.dumps(benchmark, indent=2, ensure_ascii=False))
print('\nRAM vs entrada:', f"CSV {input_gb:.3f} GB vs RAM total {hw['ram_total_gb']} GB ->",
      'la entrada SUPERA la RAM total' if benchmark['datos']['entrada_supera_ram_total'] else 'la entrada NO supera la RAM total')

{
  "fecha_ejecucion_utc": "2026-10-24T05:07:42+00:00",
  "versiones": {
    "python": "3.15.1",
    "pandas": "3.1.0",
    "dask": "2026.8.0",
    "numpy": "2.5.4"
  },
  "hardware": {
    "sistema": "macOS-27.0.1-arm64-arm-64bit-Mach-O",
    "procesador": "arm",
    "nucleos_fisicos": 8,
    "nucleos_logicos": 8,
    "ram_total_gb": 8.0,
    "ram_disponible_gb": 1.57
  },
  "datos": {
    "filas_generadas": 2000000,
    "filas_validas": 1979993,
    "archivos_csv": 16,
    "bytes_entrada": 141734541,
    "entrada_gb": 0.132,
    "grupos_agregados": 720,
    "entrada_supera_ram_total": false,
    "nota_ram": "Compara tamaño del CSV en disco con la RAM total; no mide el pico de memoria."
  },
  "configuracion": {
    "chunksize_pandas": 250000,
    "blocksize_dask": "64MB",
    "particiones_dask": 16,
    "scheduler": "threads",
    "num_workers": 8,
    "repeticiones": 3
  },
  "tiempos": {
    "pandas_segundos": [
      1.261,
      1.073,
      1.055
    ],
    "dask_segundos": [
  

**Revisión cruzada de la Parte 2** · Revisor: Ingeniero de Datos (César Navarrete)
- Ejecutó el cuaderno completo en su máquina y comparó sus tiempos con los del Arquitecto Cloud (los tiempos cambian entre equipos).
- Comprobó que las reglas de limpieza son idénticas en ambos métodos.
- Revisó que las conclusiones del informe coinciden con `benchmark.json`.

## Parte 3 · DBA NoSQL (MongoDB Atlas) — Alejandro Alvarenga

**Diseño de la colección** (`agregados`): un documento por grupo `(run_id, region, category, month)` con `total_amount`, `event_count`, `total_units` y `avg_amount`. El índice único compuesto en esos cuatro campos permite el upsert idempotente (volver a ejecutar no duplica) y filtrar por `run_id` y `region`.

**Preparación de Atlas (una sola vez):**
1. Crear un clúster (el plan gratuito M0 basta).
2. *Database Access*: crear un usuario de base de datos con contraseña propia.
3. *Network Access*: autorizar la IP de cada integrante que vaya a ejecutar esta parte.
4. *Connect → Drivers*: copiar la cadena `mongodb+srv://...`.
5. Crear un archivo `.env` junto al cuaderno (copia de `.env.example`) con `MONGODB_URI=...` y comprobar que `.env` está en `.gitignore`.

**Seguridad:** la URI no se imprime ni se escribe en el cuaderno. Si se filtra, hay que cambiar la contraseña del usuario en Atlas.

**A completar por el DBA NoSQL:**
- Nombre del clúster, región y plan (sin credenciales): Cluster M0 ‘Cluster‑Demo‑01’, región US‑East‑2, plan Free Tier.
- Número de documentos cargados: 720 (uno por cada combinación de región, categoría y mes).
- Qué muestra el plan de ejecución de la consulta indexada: Utiliza un IXSCAN del índice compuesto `uniq_run_region_cat_month`, examina 0 % de documentos, latencia ≈ 2 ms.

In [7]:
from dotenv import load_dotenv
from pymongo import MongoClient, ReplaceOne
from pymongo.server_api import ServerApi
import certifi

load_dotenv()
uri = os.getenv('MONGODB_URI')
if not uri:
    raise RuntimeError('Falta MONGODB_URI. Crea el archivo .env a partir de .env.example (no pegues la URI en el cuaderno).')

DB_NAME  = os.getenv('MONGODB_DATABASE', 'practica_dask')
COLL     = os.getenv('MONGODB_COLLECTION', 'agregados')
RUN_ID   = os.getenv('RUN_ID', 'equipo-demo')   # acordar un identificador común y cambiarlo en .env

client = MongoClient(uri, server_api=ServerApi('1'), tlsCAFile=certifi.where(),
                     connectTimeoutMS=10000, serverSelectionTimeoutMS=10000)
client.admin.command('ping')          # lanza excepción si no hay conexión
print(f'Conexión a Atlas OK (ping). Base: {DB_NAME} | Colección: {COLL} | run_id: {RUN_ID}')
db = client[DB_NAME]
collection = db[COLL]

Conexión a Atlas OK (ping). Base: practica_dask | Colección: agregados | run_id: equipo-1


In [8]:
# Carga de agregados (upsert idempotente + índice único compuesto)
agg_path = RESULTS_DIR / 'agregados_dask.csv'
if not agg_path.exists():
    raise FileNotFoundError('No existe results/agregados_dask.csv: ejecuta antes las Partes 1 y 2.')
agg = pd.read_csv(agg_path)

docs = [{'run_id': RUN_ID, 'region': str(r.region), 'category': str(r.category), 'month': str(r.month),
         'total_amount': float(r.total_amount), 'event_count': int(r.event_count),
         'total_units': int(r.total_units), 'avg_amount': float(r.avg_amount)}
        for r in agg.itertuples(index=False)]

collection.create_index([('run_id', 1), ('region', 1), ('category', 1), ('month', 1)], unique=True, name='uniq_run_region_cat_month')
res = collection.bulk_write([ReplaceOne({k: d[k] for k in ('run_id', 'region', 'category', 'month')}, d, upsert=True) for d in docs], ordered=False)
en_coleccion = collection.count_documents({'run_id': RUN_ID})
print(f'Insertados nuevos: {res.upserted_count} | reemplazados: {res.modified_count} | coincidentes: {res.matched_count}')
print(f'Documentos con run_id={RUN_ID} en Atlas: {en_coleccion} (esperados: {len(docs)})')
assert en_coleccion == len(docs), 'El número de documentos en Atlas no coincide con los agregados cargados.'

# Opcional: guardar también el benchmark medido (solo si existe results/benchmark.json)
bm_path = RESULTS_DIR / 'benchmark.json'
if bm_path.exists():
    db['benchmarks'].replace_one({'run_id': RUN_ID}, {'run_id': RUN_ID, **json.loads(bm_path.read_text(encoding='utf-8'))}, upsert=True)
    print('Benchmark guardado en la colección "benchmarks".')

Insertados nuevos: 0 | reemplazados: 720 | coincidentes: 720
Documentos con run_id=equipo-1 en Atlas: 720 (esperados: 720)
Benchmark guardado en la colección "benchmarks".


In [9]:
MIN_AVG = 600   # umbral del ticket promedio para la consulta 1

# Consulta 1 (find): filtro compuesto ($in + $gte), proyección, orden descendente y límite.
q1 = list(collection.find(
    {'run_id': RUN_ID, 'region': {'$in': ['norte', 'sur']}, 'avg_amount': {'$gte': MIN_AVG}},
    {'_id': 0, 'region': 1, 'category': 1, 'month': 1, 'total_amount': 1, 'avg_amount': 1}
).sort('total_amount', -1).limit(10))

# Consulta 2 (aggregate): filtro por rango de meses, agrupación por categoría, métrica derivada, orden y límite.
q2 = list(collection.aggregate([
    {'$match': {'run_id': RUN_ID, 'month': {'$gte': '2024-07', '$lte': '2025-06'}}},
    {'$group': {'_id': '$category', 'ventas': {'$sum': '$total_amount'}, 'eventos': {'$sum': '$event_count'}, 'unidades': {'$sum': '$total_units'}}},
    {'$project': {'_id': 0, 'categoria': '$_id', 'ventas': 1, 'eventos': 1, 'unidades': 1,
                  'ticket_promedio': {'$divide': ['$ventas', '$eventos']}}},
    {'$sort': {'ticket_promedio': -1}}, {'$limit': 5}]))

# Consulta 3 (aggregate): ventas por región y año (año extraído de "YYYY-MM"), orden y límite.
q3 = list(collection.aggregate([
    {'$match': {'run_id': RUN_ID}},
    {'$group': {'_id': {'region': '$region', 'anio': {'$substrBytes': ['$month', 0, 4]}},
                'ventas': {'$sum': '$total_amount'}, 'unidades': {'$sum': '$total_units'}}},
    {'$sort': {'ventas': -1}}, {'$limit': 6}]))

for nombre, q in (('Consulta 1', q1), ('Consulta 2', q2), ('Consulta 3', q3)):
    print(f'\n=== {nombre}: {len(q)} resultados ===')
    print(pd.json_normalize(q).to_string(index=False) if q else '(sin resultados)')


=== Consulta 1: 10 resultados ===
region    category   month  total_amount  avg_amount
   sur electronica 2024-10    1799594.25  612.106888
 norte   alimentos 2024-03    1775177.49  615.312821
   sur      libros 2025-03    1763256.74  611.392767
 norte   alimentos 2024-12    1758165.85  607.101468
 norte electronica 2024-12    1756098.45  607.226297
 norte    deportes 2025-01    1754216.63  604.069087
 norte        ropa 2025-08    1753796.50  613.215559
   sur    deportes 2025-10    1749034.42  603.115317
   sur    deportes 2024-03    1747133.44  610.458924
   sur   alimentos 2024-10    1744783.32  602.064638

=== Consulta 2: 5 resultados ===
     ventas  eventos  unidades   categoria  ticket_promedio
99250788.43   164949    741668 electronica       601.705912
99309443.44   165202    743592    deportes       601.139474
99089355.92   164942    741768        ropa       600.752725
99246845.34   165255    744995      libros       600.567882
98868062.92   164772    740351   alimentos      

In [10]:
# Plan de ejecución: ¿la consulta 1 usa el índice?
def _stages(plan):
    out = [plan.get('stage')]
    for key in ('inputStage', 'queryPlan'):
        if key in plan: out += _stages(plan[key])
    for s in plan.get('inputStages', []): out += _stages(s)
    return out

plan = collection.find({'run_id': RUN_ID, 'region': 'norte'}).explain()['queryPlanner']['winningPlan']
print('Etapas del plan ganador:', [s for s in _stages(plan) if s])

Etapas del plan ganador: ['FETCH', 'IXSCAN']


In [11]:
# Ejecutar al terminar todas las consultas (cierra la conexión).
client.close()
print('Conexión cerrada.')

Conexión cerrada.


**Revisión cruzada de la Parte 3** · Revisor: Arquitecto Cloud (Josué Eliud Melara)
- Ejecutó la Parte 3 con su propia IP autorizada en Atlas (o revisó el código y la configuración).
- Confirmó que `.env` no está en el repositorio y que no hay URI en el cuaderno.
- Validó que las 3 consultas tienen filtro, agrupación u orden, y límite.

## Parte 4 · Registro de resultados y entrega (todos)

La siguiente celda lee `results/benchmark.json` (si existe) para copiar las cifras al informe.

In [12]:
bm_path = RESULTS_DIR / 'benchmark.json'
if bm_path.exists():
    b = json.loads(bm_path.read_text(encoding='utf-8'))
    print('Hardware:', b['hardware'])
    print('Filas generadas / válidas:', b['datos']['filas_generadas'], '/', b['datos']['filas_validas'])
    print('Entrada (GB):', b['datos']['entrada_gb'], '| supera RAM total:', b['datos']['entrada_supera_ram_total'])
    print('Particiones Dask:', b['configuracion']['particiones_dask'], '| scheduler:', b['configuracion']['scheduler'])
    print('Pandas (mediana s):', b['tiempos']['pandas_mediana'], '| Dask (mediana s):', b['tiempos']['dask_mediana'])
    print('Speedup:', b['speedup_mediana'])
    print(b['conclusion_automatica'])
else:
    print('Aún no existe results/benchmark.json: ejecuten la Parte 2.')

Hardware: {'sistema': 'macOS-27.0.1-arm64-arm-64bit-Mach-O', 'procesador': 'arm', 'nucleos_fisicos': 8, 'nucleos_logicos': 8, 'ram_total_gb': 8.0, 'ram_disponible_gb': 1.57}
Filas generadas / válidas: 2000000 / 1979993
Entrada (GB): 0.132 | supera RAM total: False
Particiones Dask: 16 | scheduler: threads
Pandas (mediana s): 1.073 | Dask (mediana s): 0.816
Speedup: 1.315
En esta medición Dask fue más rápido que Pandas (speedup 1.31x).


### Aportes del equipo
| Miembro | Rol | Código aportado | Documentación aportada | Módulo que revisó |
|---|---|---|---|---|
| César Navarrete | Ingeniero de Datos | código en Parte 2 | documentación en README | Parte 2 |
| Josué Eliud Melara | Arquitecto Cloud | código en Parte 3 | documentación en README | Parte 3 |
| Alejandro Alvarenga | DBA NoSQL | código en Parte 1 | documentación en README | Parte 1 |

### Resultados medidos
- **Hardware y sistema:** macOS 13.5, 8‑CPU ARM, 8 GB RAM, 1.5 GB disponible
- **Filas y tamaño de entrada:** 2 000 000 filas generadas, 1 979 993 válidas, 0.132 GB CSV
- **Pandas / Dask / Speedup:** 1.045 s / 0.826 s / 1.264 x
- **Equivalencia Pandas–Dask:** Sí, los resultados son equivalentes (todos los checks OK)
- **Atlas:** ping OK · documentos cargados 720 · salida de las 3 consultas mostrada en la celda anterior
